In [1]:
# ==========================================
# CELL 1: IMPORTS, CONFIGURATION & BACKUP
# ==========================================
import os
import shutil
import datetime
import numpy as np
import joblib

from collections import Counter
from imblearn.over_sampling import SMOTE

from sklearn.model_selection import train_test_split
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# ── DATA PATHS ───────────────────────────────────────────
# FIRST TRAINING:
# X_PATH = r"C:\Users\kalig\OneDrive\Desktop\train_X.npy"
# Y_PATH = r"C:\Users\kalig\OneDrive\Desktop\train_y.npy"

# RETRAINING:
# after running your combiner script, point these to the combined files
X_PATH = r"C:\Users\kalig\OneDrive\Desktop\train_X.npy"
Y_PATH = r"C:\Users\kalig\OneDrive\Desktop\train_y.npy"

# ── MODEL PATHS ──────────────────────────────────────────
MODEL_SAVE_PATH  = r"C:\Users\kalig\OneDrive\Desktop\tissue_density_model_prob_ordinal.joblib"
MODEL_BACKUP_DIR = r"C:\Users\kalig\OneDrive\Desktop\model_backups"

RANDOM_STATE = 42
os.makedirs(MODEL_BACKUP_DIR, exist_ok=True)

print("=== TRAIN / RETRAIN CONFIGURATION ===")
print(f"Feature file : {X_PATH}")
print(f"Label file   : {Y_PATH}")
print(f"Model output : {MODEL_SAVE_PATH}")

# ── BACKUP OLD MODEL ─────────────────────────────────────
if os.path.exists(MODEL_SAVE_PATH):
    timestamp   = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_path = os.path.join(
        MODEL_BACKUP_DIR,
        f"tissue_density_model_prob_ordinal_{timestamp}.joblib"
    )
    shutil.copy2(MODEL_SAVE_PATH, backup_path)
    print(f" Existing model backed up to: {backup_path}")
else:
    print("No existing model found — this looks like a fresh training run.")

=== TRAIN / RETRAIN CONFIGURATION ===
Feature file : C:\Users\kalig\OneDrive\Desktop\train_X.npy
Label file   : C:\Users\kalig\OneDrive\Desktop\train_y.npy
Model output : C:\Users\kalig\OneDrive\Desktop\tissue_density_model_prob_ordinal.joblib
No existing model found — this looks like a fresh training run.


In [2]:
# ==========================================
# CELL 2: LOAD, SPLIT & BALANCE DATA
# ==========================================
print("\nLoading feature matrix and labels...")
X = np.load(X_PATH)
y = np.load(Y_PATH)

print(f"Data loaded successfully! Shape: X={X.shape}, y={y.shape}")

unique, counts = np.unique(y, return_counts=True)
print(f"Full dataset class distribution: {dict(zip(unique, counts))}")

# Split first so the test set stays untouched
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

print(f"\nBefore SMOTE — training distribution: {Counter(y_train)}")

smote = SMOTE(
    sampling_strategy='auto',
    random_state=RANDOM_STATE,
    k_neighbors=7
)

X_train_balanced, y_train_balanced = smote.fit_resample(X_train, y_train)

print(f"After  SMOTE — training distribution: {Counter(y_train_balanced)}")

real_train_images  = X_train.shape[0]
total_train_images = X_train_balanced.shape[0]
synthetic_images   = total_train_images - real_train_images
test_images        = X_test.shape[0]

print(f"\n{'='*50}")
print("DATA SUMMARY")
print(f"{'='*50}")
print(f"Original training images   : {real_train_images}")
print(f"Synthetic images added     : {synthetic_images}")
print(f"Total training images used : {total_train_images}")
print(f"Test images                : {test_images}")
print(f"Features per image         : {X_train_balanced.shape[1]}")
print(f"{'='*50}")


Loading feature matrix and labels...
Data loaded successfully! Shape: X=(9999, 1792), y=(9999,)
Full dataset class distribution: {np.int64(1): np.int64(1172), np.int64(2): np.int64(4282), np.int64(3): np.int64(3991), np.int64(4): np.int64(554)}

Before SMOTE — training distribution: Counter({np.int64(2): 3425, np.int64(3): 3193, np.int64(1): 938, np.int64(4): 443})
After  SMOTE — training distribution: Counter({np.int64(1): 3425, np.int64(2): 3425, np.int64(3): 3425, np.int64(4): 3425})

DATA SUMMARY
Original training images   : 7999
Synthetic images added     : 5701
Total training images used : 13700
Test images                : 2000
Features per image         : 1792


In [3]:
# ==========================================
# CELL 3: PROBABILITY-BASED ORDINAL ADABOOST
# ==========================================
class ProbOrdinalAdaBoost:
    """
    Ordinal AdaBoost using K-1 binary threshold models.

    For ordered classes [1,2,3,4], it trains:
      P(y > 1), P(y > 2), P(y > 3)

    Final class probabilities are reconstructed as:
      P(y=1) = 1 - q1
      P(y=2) = q1 - q2
      P(y=3) = q2 - q3
      P(y=4) = q3
    where qj = P(y > threshold_j)
    """
    def __init__(self, n_estimators=150, random_state=None):
        self.n_estimators = n_estimators
        self.random_state = random_state
        self.models = []
        self.classes_ = None

    def fit(self, X: np.ndarray, y: np.ndarray) -> None:
        self.classes_ = np.sort(np.unique(y))
        K = len(self.classes_)

        if K < 3:
            raise ValueError(f"ProbOrdinalAdaBoost requires at least 3 ordered classes, got {K}.")

        self.models = []

        for i in range(K - 1):
            threshold = self.classes_[i]
            y_binary = (y > threshold).astype(int)

            model = AdaBoostClassifier(
                estimator=DecisionTreeClassifier(max_depth=1),
                n_estimators=self.n_estimators,
                random_state=self.random_state
            )
            model.fit(X, y_binary)
            self.models.append(model)

            print(f" Binary classifier {i+1}/{K-1} trained "
                  f"(threshold task: y > {threshold})")

    def predict_threshold_proba(self, X: np.ndarray) -> np.ndarray:
        """
        Returns q with shape (n_samples, K-1),
        where q[:, i] = P(y > threshold_i)
        """
        q = np.column_stack([
            model.predict_proba(X)[:, 1] for model in self.models
        ])

        # Enforce ordinal monotonicity:
        # P(y > 1) >= P(y > 2) >= P(y > 3)
        for i in range(1, q.shape[1]):
            q[:, i] = np.minimum(q[:, i-1], q[:, i])

        return q

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        q = self.predict_threshold_proba(X)

        n = X.shape[0]
        K = len(self.classes_)
        probs = np.zeros((n, K), dtype=float)

        # Convert threshold probabilities to class probabilities
        probs[:, 0] = 1.0 - q[:, 0]
        for i in range(1, K - 1):
            probs[:, i] = q[:, i - 1] - q[:, i]
        probs[:, K - 1] = q[:, K - 2]

        # Numerical cleanup
        probs = np.clip(probs, 0.0, 1.0)
        row_sums = probs.sum(axis=1, keepdims=True)
        row_sums[row_sums == 0] = 1.0
        probs /= row_sums

        return probs

    def predict(self, X: np.ndarray) -> np.ndarray:
        probs = self.predict_proba(X)
        return self.classes_[np.argmax(probs, axis=1)]

In [4]:
# ==========================================
# CELL 4: TRAIN, EVALUATE & SAVE
# ==========================================
print("\nTraining Probability-Based Ordinal AdaBoost...")
model = ProbOrdinalAdaBoost(n_estimators=150, random_state=RANDOM_STATE)
model.fit(X_train_balanced, y_train_balanced)

print("\nEvaluating on test set...")
y_pred  = model.predict(X_test)
y_proba = model.predict_proba(X_test)

acc = accuracy_score(y_test, y_pred)
cm  = confusion_matrix(y_test, y_pred)

print(f"\n{'='*50}")
print("PERFORMANCE REPORT")
print(f"{'='*50}")
print(f"Images model trained on    : {total_train_images}")
print(f"  ↳ Real images            : {real_train_images}")
print(f"  ↳ SMOTE synthetic        : {synthetic_images}")
print(f"Images tested on           : {test_images}")
print(f"Overall accuracy           : {acc:.4f}")
print(f"{'='*50}")

print("\nPer-class breakdown:")
print(classification_report(
    y_test,
    y_pred,
    target_names=[
        'Tissue Density Category 1',
        'Tissue Density Category 2',
        'Tissue Density Category 3',
        'Tissue Density Category 4'
    ]
))

print("Confusion Matrix:")
header = "              " + "  ".join(f"Pred {i+1}" for i in range(4))
print(header)
for i, row in enumerate(cm):
    print(f"Actual {i+1}     " + "  ".join(f"{v:6d}" for v in row))

print("\nFirst 5 class-probability rows:")
print(y_proba[:5])

print(f"\nSaving model to {MODEL_SAVE_PATH}...")
joblib.dump(model, MODEL_SAVE_PATH)
print(" Done! Probability-based ordinal model saved.")


Training Probability-Based Ordinal AdaBoost...
 Binary classifier 1/3 trained (threshold task: y > 1)
 Binary classifier 2/3 trained (threshold task: y > 2)
 Binary classifier 3/3 trained (threshold task: y > 3)

Evaluating on test set...

PERFORMANCE REPORT
Images model trained on    : 13700
  ↳ Real images            : 7999
  ↳ SMOTE synthetic        : 5701
Images tested on           : 2000
Overall accuracy           : 0.1720

Per-class breakdown:
                           precision    recall  f1-score   support

Tissue Density Category 1       0.22      1.00      0.36       234
Tissue Density Category 2       0.00      0.00      0.00       857
Tissue Density Category 3       0.00      0.00      0.00       798
Tissue Density Category 4       0.12      0.99      0.21       111

                 accuracy                           0.17      2000
                macro avg       0.08      0.50      0.14      2000
             weighted avg       0.03      0.17      0.05      2000

Confus

c:\Users\kalig\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\kalig\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\kalig\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\metrics\_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(ave

In [5]:
# ==========================================
# CELL 5: PREDICTION UTILITY
# ==========================================
def predict_new(X_new: np.ndarray):
    """
    Returns:
      preds           -> predicted tissue density class labels
      class_probs     -> P(class = 1..4)
      threshold_probs -> P(y > 1), P(y > 2), P(y > 3)
    """
    preds = model.predict(X_new)
    class_probs = model.predict_proba(X_new)
    threshold_probs = model.predict_threshold_proba(X_new)
    return preds, class_probs, threshold_probs

print(" Prediction utility ready.")
print("Use: preds, class_probs, threshold_probs = predict_new(X_new)")

 Prediction utility ready.
Use: preds, class_probs, threshold_probs = predict_new(X_new)
